# 06 — validation lock과 finalist 추가 seed

> **선택 실행용 실험 기반 코드입니다.** 최신 최종 실험 명세 전체를 구현한 상태는 아닙니다. 실제 본 실험 전 [`docs/ml/README.md`의 구현 차이](../../../docs/ml/README.md#implementation-gaps)를 확인하고 필요한 보완을 완료하세요. 이 노트북의 설정을 최종 연구 기준으로 사용하지 않습니다.

먼저 네 seed-42 full run으로 provisional top-2를 고릅니다. 이후 아래 `METHOD`, `SEED`를 바꿔 finalist×{17,73} 네 run을 **세션별 한 번씩** 실행하세요. 네 run이 모두 게시된 뒤 마지막 셀에서 final lock을 만듭니다. test와 RouterArena는 읽지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
archive='/content/drive/MyDrive/LLM_Router/colab-inputs/routing-objective-colab-code.tar.gz'
!rm -rf /content/LLM_Router
!tar -xzf "{archive}" -C /content
%cd /content/LLM_Router
!pip -q install -r ml/requirements-comparison.lock
!pip -q install 'transformers==4.48.3' 'safetensors>=0.4'
!pip -q install --no-deps -e ml
import json, os, pathlib, shutil, subprocess, torch
ROOT=pathlib.Path('/content/drive/MyDrive/LLM_Router/routing-objective-comparison/v1')


In [ ]:
for method in ('ordinal','pointwise','pairwise','regret'):
    if not (ROOT/f'runs/{method}/seed-42/_SUCCESS').is_file(): raise RuntimeError(f'{method} seed-42 full run이 없습니다.')
PROVISIONAL=ROOT/'selection/provisional-lock.json'
PROVISIONAL.parent.mkdir(parents=True,exist_ok=True)
subprocess.run(['routing-objective-comparison','--config','ml/config/routing_objective_comparison_v1.json','lock-validation','--run-root',str(ROOT/'runs'),'--output-path',str(PROVISIONAL),'--stage','provisional'],check=True)
lock=json.loads(PROVISIONAL.read_text())
print('finalists:',lock['finalists'])
print('provisional winner:',lock['winner_method'])


## 한 세션에 하나의 추가 seed 실행
`METHOD`는 출력된 finalist 중 하나, `SEED`는 17 또는 73으로 바꾸세요. 이미 완료된 조합은 다시 실행하지 않습니다.


In [ ]:
METHOD = lock['finalists'][0]  # 필요할 때 두 번째 finalist로 변경
SEED = 17                     # 다음 세션에는 73으로 변경
if METHOD not in lock['finalists'] or SEED not in (17,73): raise ValueError('finalist와 seed를 확인하세요.')
FINAL=ROOT/f'runs/{METHOD}/seed-{SEED}'
if (FINAL/'_SUCCESS').is_file():
    print('이미 완료:',FINAL)
else:
    assert torch.cuda.is_available(), '추가 seed 학습에는 T4 GPU 런타임이 필요합니다.'
    LOCAL=pathlib.Path('/content/objective-data')
    if LOCAL.exists(): shutil.rmtree(LOCAL)
    shutil.copytree(ROOT/'dataset',LOCAL)
    OUT=pathlib.Path('/content/finalist-run')
    if OUT.exists(): shutil.rmtree(OUT)
    subprocess.run(['routing-objective-comparison','--config','ml/config/routing_objective_comparison_v1.json','train','--method',METHOD,'--training-path',str(LOCAL/'train.parquet'),'--validation-path',str(LOCAL/'validation.parquet'),'--outcome-manifest-path',str(LOCAL/'manifest.json'),'--output-dir',str(OUT),'--seed',str(SEED)],check=True)
    staging=pathlib.Path(str(FINAL)+'.staging')
    if staging.exists(): shutil.rmtree(staging)
    staging.parent.mkdir(parents=True,exist_ok=True); shutil.copytree(OUT,staging)
    os.replace(staging,FINAL)
    print('게시 완료:',FINAL)


In [ ]:
for method in lock['finalists']:
    print(method,{seed:(ROOT/f'runs/{method}/seed-{seed}/_SUCCESS').is_file() for seed in (17,73)})
print('네 조합이 모두 True일 때만 다음 셀을 실행하세요.')


In [ ]:
FINAL_LOCK=ROOT/'selection/final-lock.json'
subprocess.run(['routing-objective-comparison','--config','ml/config/routing_objective_comparison_v1.json','lock-validation','--run-root',str(ROOT/'runs'),'--output-path',str(FINAL_LOCK),'--stage','final'],check=True)
final_lock=json.loads(FINAL_LOCK.read_text())
print('winner:',final_lock['winner_method'])
print(json.dumps(final_lock['methods'],indent=2,ensure_ascii=False))
